# Laboratorio 4 — API de Hilos
## Análisis de Rendimiento

| Campo | Valor |
|---|---|
| **CPU** | AMD Ryzen 5 3600 |
| **Núcleos físicos** | 6 |
| **Hilos lógicos (SMT)** | 12 |
| **n (pasos de integración)** | 2 000 000 000 |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.dpi'] = 120

## Sección 1: Análisis del Cálculo de π

### Actividad 1 — Tiempo Serial (Ts)

```bash
./pi_s 2000000000
```

In [ ]:
# ./pi_s 2000000000  →  time = 2.806809 s
Ts = 2.806809
n  = 2_000_000_000
print(f'Ts = {Ts:.6f} s  (n = {n:,})')

### Actividad 2 — Tiempos Paralelos Tp(N)

```bash
./pi_p 2000000000 <N>
```

N ∈ {1, 2, 4, 8, 12}  (hasta 2× los 6 núcleos físicos)

In [ ]:
threads = [1,        2,        4,        8,        12      ]
Tp      = [2.776728, 1.513339, 0.849189, 0.571811, 0.482122]

for N, tp in zip(threads, Tp):
    print(f'  N={N:2d}  Tp = {tp:.6f} s')

### Actividad 3 — Tabla de Métricas de Rendimiento

In [ ]:
speedup    = [Ts / tp for tp   in Tp                   ]
efficiency = [s  / N  for s, N in zip(speedup, threads)]

df = pd.DataFrame({
    'N (Hilos)'             : threads,
    'Tp (s)'                : Tp,
    'Speedup (Ts/Tp)'       : speedup,
    'Eficiencia (Speedup/N)': efficiency,
}).set_index('N (Hilos)')

df.style.format({
    'Tp (s)'                : '{:.6f}',
    'Speedup (Ts/Tp)'       : '{:.4f}',
    'Eficiencia (Speedup/N)': '{:.4f}',
})

### Actividad 4 — Gráfico de Speedup

In [ ]:
ideal_x = list(range(1, max(threads) + 1))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(threads, speedup, 'bo-', linewidth=2, markersize=8, label='Speedup medido')
ax.plot(ideal_x, ideal_x, 'r--', linewidth=1.5, alpha=0.7, label='Speedup ideal')

for N, s in zip(threads, speedup):
    ax.annotate(f'{s:.2f}', xy=(N, s), xytext=(4, 6),
                textcoords='offset points', fontsize=9)

ax.set_xlabel('N — Número de Hilos')
ax.set_ylabel('Speedup  (Ts / Tp)')
ax.set_title('Speedup vs. Número de Hilos — Cálculo de π  (n = 2 000 000 000)')
ax.set_xticks(threads)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('speedup_pi.png', dpi=150, bbox_inches='tight')
plt.show()

### Actividad 5 — Análisis de Resultados

#### 5.1 Comparación Tp(1) vs Ts — Overhead de pthreads

El tiempo paralelo con un solo hilo **Tp(1) = 2.7767 s** es prácticamente igual al tiempo
serial **Ts = 2.8068 s**, con una diferencia de apenas **~1.07 %**.
Esta pequeña discrepancia representa el *overhead* de pthreads: las llamadas a
`pthread_create` y `pthread_join`, más la asignación dinámica de `ThreadArgs`.
Con un único hilo no existe paralelismo real; el costo adicional es sólo de gestión del hilo.

#### 5.2 Speedup máximo alcanzado

El **speedup máximo** obtenido fue **5.82×** con **N = 12 hilos**, sobre una CPU de
**6 núcleos físicos** (AMD Ryzen 5 3600, SMT 2×):

- Se alcanzó el **97 %** del speedup teórico limitado por los núcleos físicos (6×).
- Al escalar de 8 → 12 hilos (zona SMT) la ganancia marginal es reducida: dos hilos
  lógicos comparten las mismas unidades de ejecución del núcleo físico.
- La **Ley de Amdahl** impone un techo: incluso una fracción serial mínima
  (reducción final, gestión de hilos) limita el speedup máximo alcanzable.

#### 5.3 Tendencia de la Eficiencia

La eficiencia **decrece monótonamente** al aumentar N, pasando de ~1.01 (N=1)
hasta ~0.49 (N=12). Las causas principales son:

1. **Overhead de sincronización:** `pthread_create` y `pthread_join` consumen
   tiempo creciente con N sin aportar cómputo útil.
2. **Saturación de ancho de banda de memoria:** los N hilos compiten por el
   mismo bus RAM al acceder a sus sub-rangos del dominio de integración.
3. **Contención de caché L3:** más hilos activos aumentan la probabilidad de
   *cache misses* y el tráfico de coherencia.
4. **Rendimientos decrecientes del SMT (N=8→12):** los hilos lógicos comparten
   recursos físicos del núcleo, reduciendo la ganancia neta por hilo adicional.

## Sección 2: Análisis del Generador de Fibonacci

### 1. Resultados de Ejecución — `./fibonacci 15`

In [ ]:
output_fib15 = """F(0) = 0
F(1) = 1
F(2) = 1
F(3) = 2
F(4) = 3
F(5) = 5
F(6) = 8
F(7) = 13
F(8) = 21
F(9) = 34
F(10) = 55
F(11) = 89
F(12) = 144
F(13) = 233
F(14) = 377"""
print(output_fib15)

### 2. Comparación Serial vs. Multihilo (N = 200 000)

Comando ejecutado:
```bash
./fibonacci   200000   # versión multihilo
./fibonacci_s 200000   # versión serial (sin hilos)
```

In [ ]:
import pandas as pd

T_serial = 0.001382889
T_hilo   = 0.001378942
N        = 200_000

df_fib = pd.DataFrame({
    'Versión'   : ['Serial (fibonacci_s)', 'Multihilo (fibonacci)'],
    'N'         : [N, N],
    'Tiempo (s)': [T_serial, T_hilo],
    'Overhead'  : [0.0, T_hilo - T_serial],
}).set_index('Versión')

df_fib.style.format({
    'N'         : '{:,}',
    'Tiempo (s)': '{:.9f}',
    'Overhead'  : '{:+.9f}',
})

### 3. Análisis del Diseño

#### 3.1 Mecanismo de transferencia de datos al hilo trabajador

Se utiliza una estructura `ThreadArgs` definida en [fibonacci.c](punto2/fibonacci.c):

```c
typedef struct {
    long long *arr;   // puntero al arreglo compartido (asignado en main con malloc)
    int        n;     // número de elementos a calcular
} ThreadArgs;
```

El flujo de transferencia es el siguiente:

1. **`main` asigna memoria dinámica** con `malloc(n * sizeof(long long))` y guarda el
   puntero en `arr`.
2. **Se inicializa la estructura** en el stack de `main`:
   `ThreadArgs args = { arr, n };`
3. **`pthread_create` recibe `&args`** (convertido implícitamente a `void *`) como
   cuarto argumento, que es el parámetro que le pasará al hilo trabajador.
4. **Dentro de `fib_worker`**, el parámetro `void *arg` se convierte de vuelta:
   `ThreadArgs *a = (ThreadArgs *)arg;`
   y se accede a los datos con `a->arr` y `a->n`.

Dado que ambos hilos comparten el mismo espacio de direcciones (modelo de **memoria
compartida**), el worker escribe directamente sobre el mismo arreglo físico que `main`
creó. No existe copia de datos: el puntero apunta a la misma región de heap.

#### 3.2 Rol de `pthread_join` como mecanismo de sincronización

En este programa, `pthread_join(worker, NULL)` cumple tres roles críticos:

| Rol | Descripción |
|---|---|
| **Espera activa** | Bloquea a `main` hasta que `fib_worker` ejecute `return NULL` |
| **Barrera de memoria** | Garantiza que todas las escrituras del worker en `arr[]` sean visibles para `main` antes de continuar |
| **Prevención de condición de carrera** | Sin `pthread_join`, `main` podría leer valores no inicializados de `arr[]` |

**¿Qué pasaría sin `pthread_join`?**

```c
pthread_create(&worker, NULL, fib_worker, &args);
// sin pthread_join:
for (int i = 0; i < n; i++)
    printf("F(%d) = %lld\n", i, arr[i]);  // ← lectura de datos incompletos / basura
```

`main` avanzaría inmediatamente al `for` de impresión mientras `fib_worker` todavía
está llenando el arreglo. El resultado sería impredecible: algunos valores estarían
calculados, otros serían cero u "basura" de memoria.

**Flujo correcto con `pthread_join`:**

```
main                          fib_worker
  │                               │
  ├── malloc(arr)                 │
  ├── pthread_create ────────────►│
  ├── pthread_join (bloqueado)    ├── arr[0]=0, arr[1]=1
  │        │                     ├── arr[i]=arr[i-1]+arr[i-2]
  │        │                     │        ...
  │        │                     ├── return NULL
  │◄───────┘ (desbloquea)        │
  ├── imprime arr[0..n-1]        │
  └── free(arr)
```

La Fibonacci es inherentemente **secuencial** (cada valor depende de los dos anteriores),
por lo que el uso de un hilo trabajador no aporta speedup computacional. El valor del
patrón reside en demostrar la separación de responsabilidades: `main` gestiona memoria
y presentación, el worker realiza el cómputo, y `pthread_join` garantiza la
sincronización entre ambos.